# Notebook 07: Prompt Engineering

In this notebook, we will learn how to **engineer prompts** to get better results from our RAG pipeline.

## What You Will Learn

- What prompt engineering is
- Techniques for better prompts
- How to prevent hallucinations
- How to force structured output
- How to handle edge cases

## What is Prompt Engineering?

**Prompt engineering** is the art of writing instructions that help LLMs understand exactly what you want.

Good prompts:
- Are clear and specific
- Provide examples
- Set boundaries and constraints
- Define the expected output format

## Why is Prompt Engineering Critical in RAG?

In RAG, the prompt must:
1. Tell the model to use ONLY the retrieved context
2. Prevent hallucinations (making up facts)
3. Handle cases where the answer isn't in the context
4. Produce consistent, structured output

## Step 1: Basic Prompt vs. Engineered Prompt

Let's compare a simple prompt with a well-engineered one.

In [1]:
# Basic (weak) prompt
basic_prompt = """
Here is some context: {context}

Question: {question}

Answer the question.
"""

# Engineered (strong) prompt
engineered_prompt = """
You are a precise document analyst. Your task is to answer questions based ONLY on the provided document excerpts.

## Document Context:
{context}

## User Question:
{question}

## Strict Rules:
1. Answer using ONLY information found in the Document Context above.
2. If the answer is not in the context, respond EXACTLY with: "I cannot find the answer in the provided document."
3. Do not use your general knowledge to answer.
4. Do not speculate or guess.
5. If the context is insufficient, state what information is missing.

## Your Answer:
"""

print("Basic prompt length:", len(basic_prompt))
print("Engineered prompt length:", len(engineered_prompt))

Basic prompt length: 77
Engineered prompt length: 546


## Step 2: Setup and Test

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Setup (reusing previous code)
loader = PyPDFLoader("../data/sample.pdf")
pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(pages)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

llm = OllamaLLM(model="mistral", temperature=0.1, num_ctx=4096)

def format_docs(docs):
    return "\n\n".join([
        f"[{i+1}] (Page {doc.metadata.get('page', '?')})\n{doc.page_content}"
        for i, doc in enumerate(docs)
    ])

print("Setup complete!")

C:\Users\Ahmed\AppData\Local\Temp\ipykernel_26096\1613994130.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Setup complete!


## Step 3: Compare Basic vs. Engineered Prompts

In [3]:
# Build chains with both prompts
basic_template = PromptTemplate.from_template(basic_prompt)
basic_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | basic_template
    | llm
)

engineered_template = PromptTemplate.from_template(engineered_prompt)
engineered_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | engineered_template
    | llm
)

print("Both chains built!")

Both chains built!


## Step 4: Test with a Question Inside the Document

In [4]:
question = "What is the main topic of this document?"

print(f"Question: {question}")
print(f"\n{'=' * 60}")
print("BASIC PROMPT RESPONSE:")
print(f"{'=' * 60}")
basic_answer = basic_chain.invoke(question)
print(basic_answer)

print(f"\n{'=' * 60}")
print("ENGINEERED PROMPT RESPONSE:")
print(f"{'=' * 60}")
engineered_answer = engineered_chain.invoke(question)
print(engineered_answer)

Question: What is the main topic of this document?

BASIC PROMPT RESPONSE:
 The main topic of this document is Artificial Intelligence (AI), with specific focus on Natural Language Processing (NLP) and Deep Learning. It discusses various tasks related to NLP such as tokenization, sentiment analysis, named entity recognition, text summarization, and machine translation. Additionally, it touches upon Deep Learning, its applications in computer vision, and some of the ethical concerns associated with AI like privacy, job displacement, transparency, and safety.

ENGINEERED PROMPT RESPONSE:
 The main topic of this document is Artificial Intelligence (AI). Specifically, it discusses Natural Language Processing (NLP) in Chapter 4 and Deep Learning in Chapter 3.


## Step 5: Test with an Out-of-Domain Question

In [5]:
# This question is NOT in the document
question = "What is the capital of France?"

print(f"Question: {question}")
print(f"\n{'=' * 60}")
print("BASIC PROMPT RESPONSE:")
print(f"{'=' * 60}")
basic_answer = basic_chain.invoke(question)
print(basic_answer)

print(f"\n{'=' * 60}")
print("ENGINEERED PROMPT RESPONSE:")
print(f"{'=' * 60}")
engineered_answer = engineered_chain.invoke(question)
print(engineered_answer)

Question: What is the capital of France?

BASIC PROMPT RESPONSE:
 The capital of France is Paris.

ENGINEERED PROMPT RESPONSE:
 I cannot find the answer in the provided document.


## Step 6: Advanced Prompt Techniques

Here are several techniques to improve your prompts:

In [6]:
# Technique 1: Role-based prompting
role_prompt = """
You are a {role} with expertise in {expertise}.

Your task is to analyze the following document context and answer the user's question.

## Role: {role}
## Expertise: {expertise}

## Document Context:
{context}

## Question:
{question}

## Response Format:
- Answer: [Your answer based on the context]
- Confidence: [High/Medium/Low] based on how well the context supports the answer
- Source: [Which part of the context supports this]

## Strict Rules:
- Use ONLY the provided context
- If unsure, say "I cannot determine this from the provided document."

## Your Analysis:
"""

role_template = PromptTemplate.from_template(role_prompt)

# Create a chain with role-based prompting
role_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()} | role_template | llm
)

# Test it
question = "What is the main topic of this document?"
answer = role_chain.invoke({
    "context": "placeholder",  # Will be replaced by retriever
    "question": question,
    "role": "Document Analyst",
    "expertise": "Reading comprehension and information extraction"
})
# Note: This needs the full chain setup, shown for concept demonstration
print("Role-based prompt created (concept demonstrated)")

AttributeError: 'dict' object has no attribute 'replace'

In [ ]:
# Technique 2: Few-shot prompting (providing examples)
few_shot_prompt = """
You are a document QA assistant. Answer questions using ONLY the provided context.

## Examples of good answers:

Context: "Python is a programming language created by Guido van Rossum."
Question: "Who created Python?"
Answer: "Python was created by Guido van Rossum."

Context: "The Eiffel Tower is 330 meters tall."
Question: "How tall is the Eiffel Tower?"
Answer: "The Eiffel Tower is 330 meters tall."

Context: "The Eiffel Tower is 330 meters tall."
Question: "What is the capital of France?"
Answer: "I cannot find the answer in the provided document."

## Now answer the user's question:

## Context:
{context}

## Question:
{question}

## Answer:
"""

few_shot_template = PromptTemplate.from_template(few_shot_prompt)
few_shot_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | few_shot_template
    | llm
)

print("Few-shot prompt created!")

In [7]:
# Technique 3: Chain-of-thought prompting

# Ask the model to think step by step
cot_prompt = """
You are a document analyst. Answer questions using ONLY the provided context.

## Context:
{context}

## Question:
{question}

## Instructions:
First, think step by step:
1. Read the question carefully
2. Search through the context for relevant information
3. Determine if the answer is present
4. Formulate a clear answer

Then provide your final answer.

## Step-by-step reasoning:

## Final Answer:
"""

cot_template = PromptTemplate.from_template(cot_prompt)
cot_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | cot_template
    | llm
)

print("Chain-of-thought prompt created!")

Chain-of-thought prompt created!


## Prompt Engineering Best Practices

| Technique | Purpose | When to Use |
|-----------|---------|-------------|
| Role-based | Gives the model a perspective | When you need domain-specific answers |
| Few-shot | Shows the model what you expect | When output format matters |
| Chain-of-thought | Forces reasoning before answering | For complex questions |
| Negative instructions | Prevents unwanted behavior | Always use in RAG |
| Output format | Controls response structure | When you need structured output |
| Temperature control | Balances creativity vs accuracy | Use low temp (0.1) for RAG |

## Key Takeaways

1. **Well-engineered prompts** significantly improve RAG quality
2. **Negative instructions** ("do not hallucinate") are crucial in RAG
3. **Few-shot examples** help the model understand the expected format
4. **Chain-of-thought** improves accuracy for complex questions
5. **Low temperature** (0.1) is best for factual RAG tasks

## Next Steps

Proceed to **Notebook 08: Output Parser** to structure the LLM's output.